# Multi-Tool Agent

Bu projede gelen soruyu aritmetik / belge arama / sözlük diye ayıran küçük bir agent yazacağım.


In [ ]:
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt


### Data


In [ ]:
df=pd.read_csv('data/bbc-news-data.csv',sep='\t')
textcol='content' if 'content' in df.columns else df.columns[-1]
df[textcol]=df[textcol].fillna('')
df=df.head(500)

ornek=pd.DataFrame({
    'soru':['2+5 nedir','what is inflation',' Merhaba','premier league news'],
    'arac':['hesap','arama','selam','arama']
})
ornek


### EDA


In [ ]:
ornek['arac'].value_counts()


### Görselleştirme


In [ ]:
ornek['arac'].value_counts().plot(kind='bar')
plt.show()


### Boş veri


In [ ]:
ornek['soru']=ornek['soru'].fillna('')


### Feature Engineering + 3 model (araç seçimi)


In [ ]:
rows=[]
for t in ['topla hesap 3+4','selam merhaba','search football','2*8 nedir','hi there','news about bank']:
    if any(k in t for k in ['+','*','hesap','nedir']) and any(ch.isdigit() for ch in t):
        lab='hesap'
    elif any(k in t for k in ['selam','merhaba','hi']):
        lab='selam'
    else:
        lab='arama'
    rows.append((t,lab))
labdf=pd.DataFrame(rows,columns=['soru','arac'])
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
vec=TfidfVectorizer()
x=vec.fit_transform(labdf['soru']); y=labdf['arac']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)


### 3 Model


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

for ad,m in [('NB',MultinomialNB()),('LogReg',LogisticRegression(max_iter=200)),('SVC',LinearSVC())]:
    m.fit(x_train,y_train)
    print(ad,accuracy_score(y_test,m.predict(x_test)))


In [ ]:
def hesap(s):
    s=s.replace('nedir','').replace('?','')
    return eval(s,{'__builtins__':{}})

def ara(q):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    v=TfidfVectorizer(stop_words='english')
    X=v.fit_transform(df[textcol])
    i=cosine_similarity(v.transform([q]),X).ravel().argmax()
    return df.iloc[i][textcol][:250]

def agent(q):
    ql=q.lower()
    if any(ch.isdigit() for ch in q) and any(op in q for op in '+-*/'):
        return ('hesap',hesap(q))
    if any(k in ql for k in ['selam','merhaba','hi']):
        return ('selam','merhaba')
    return ('arama',ara(q))

print(agent('3+4'))
print(agent('hello'))


In [ ]:
import joblib
joblib.dump({'ok':True},'../../models/agent_multitool.joblib')


### Sonuç

3 araçlı küçük agent çalışıyor. Set küçük olduğu için intent modeli zayıf, kural + arama daha sağlam. Hedefi temel olarak tutturdum.
